# Disturbance robustness: the tube's RPI certificate, measured

`01_OVERVIEW.md` §1.6 is honest about a hard limit: a discrete-time CBF row by
itself confers no robustness to disturbance. The tube MPC-CBF controller
(`tube_mpc_cbf_solver`) closes that gap with an offline RPI set `Omega`
(`08_TUBE.md`): the closed-loop error `e_k = x_k - z_k` stays inside `Omega`
for all `k` provided `e_0` starts there. That guarantee is a **set-containment
certificate**, not a trajectory promise — and this notebook turns the honesty
about it into three measurements, on the same `N = 8` tube fixture used by
`test_tube_mpc_robustness.cpp`:

1. **The RPI set.** `Omega` projected onto `(px,py)` and `(vx,vy)`, with
   10 000 sampled steady-state closed-loop errors overlaid. **Assert 100 %
   containment.** If the cloud escapes `Omega`, the RPI computation is wrong
   and everything downstream is decoration.
2. **Magnitude sweep.** `w_max` geometric sweep, 50 seeds, nominal vs tube:
   `min_k h`, violation rate, cost. The tube must hold `min h >= 0` up to the
   design `w_max` **and then also fail**.
3. **Conservatism cost.** Path length and time-to-goal versus nominal at *zero*
   disturbance — the honest counterweight, reported in the README.

Every number below is a measurement; every claim is an `assert`.


In [ ]:
# --- Imports + determinism ---------------------------------------------------
# Analysis ground rules (12_ANALYSIS.md): fixed seed printed first, every
# claim an assert, figures to analysis/figures/, CSV next to the notebook.
RNG_SEED = 0xC0FFEE
print(f"RNG_SEED = 0x{RNG_SEED:X}")

import os
import sys
import csv
import warnings
from pathlib import Path

import numpy as np
from scipy.optimize import linprog
import matplotlib

matplotlib.use("Agg")  # headless; CI has no display
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")


def find_repo_root(start=None):
    d = (start or Path.cwd()).resolve()
    for _ in range(6):
        if (d / "codegen").is_dir():
            return d
        if d.parent == d:
            break
        d = d.parent
    raise RuntimeError("repo root not found")


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT))

from mpc_cbf_unified.test import test_recursive_feasibility as rf  # noqa: E402
from codegen.generate_tube_solver import (  # noqa: E402
    compute_offline_sets, build_tube_ocp, tube_param_count, _rakovic_rpi,
)
from acados_template import AcadosOcpSolver  # noqa: E402

HERE = REPO_ROOT / "analysis"
FIGDIR = HERE / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)

# --- Tube fixture constants ---------------------------------------------------
# The tube fixture (test/test_tube_mpc_robustness.cpp) uses its own radii and
# disturbance box, which DIFFER from test_recursive_feasibility.py (rf uses
# EGO_RADIUS=0.15 / R_EFF=0.4). Do NOT import R_EFF / EGO_RADIUS from rf here.
# These are the C++ fixture values, mirroring fixtureWHalfWidths() etc.
DT = rf.DT                      # 0.1
N = 8                           # horizon (mpc_config_.horizon)
N_OBSTACLES = rf.N_OBSTACLES    # 8
GAMMA = rf.GAMMA                # 0.3
GOAL = rf.GOAL                  # [1, 1, 0, 0]
OBST_POS = np.array([0.5, 0.5, 0.0])
OBST_RADIUS = 0.2
EGO_RADIUS = 0.1                # tube fixture ego radius
SAFETY_MARGIN = 0.05
R_EFF = OBST_RADIUS + EGO_RADIUS + SAFETY_MARGIN  # 0.35
W_HALF = np.array([0.005, 0.005, 0.02, 0.02])     # fixtureWHalfWidths()
RNG_SEED = rf.RNG_SEED          # 0xC0FFEE

# Cost weights: the C++ test fixture sets Qf = Q = [10,10,1,1] for its safety
# tests, but the shipped configuration (mpc_cbf_params.yaml) uses
# Qf = 10 Q = [100,100,10,10]. This notebook uses the SHIPPED weights (the
# build_tube_ocp defaults) because item 3 measures time-to-goal, and with
# Qf = Q the tube idles at |x| ~ 0.145 and never reaches the goal (item 3 is
# then undefined). The safety/containment numbers (items 1-2) are insensitive
# to this choice (see the closing cell).


def barrier_h(x, p=OBST_POS, r_eff=R_EFF):
    """Squared-distance barrier, position indices (0,1): ||p - p_obs||^2 - r^2."""
    dx = x[0] - p[0]
    dy = x[1] - p[1]
    return dx * dx + dy * dy - r_eff * r_eff


def advance(x, u, w):
    """Double-integrator step + disturbance: x_next = A x + B u + w."""
    return rf._closed_loop_step(x, u) + w


def worst_case_w(x, p=OBST_POS, half=W_HALF):
    """WorstCaseW() from the C++ fixture: each axis pushes toward the obstacle."""
    to_obs = p[:2] - x[:2]
    return np.array([
        half[0] if to_obs[0] > 0 else -half[0],
        half[1] if to_obs[1] > 0 else -half[1],
        half[2] if to_obs[0] > 0 else -half[2],
        half[3] if to_obs[1] > 0 else -half[3],
    ])


class _silence_io:
    """acados QP-failure diagnostics go to fd 1 (print_level=0 does not
    silence them); redirect both fds to /dev/null around each solve."""

    def __enter__(self):
        self._fd1 = os.dup(1)
        self._fd2 = os.dup(2)
        self._devnull = os.open(os.devnull, os.O_WRONLY)
        os.dup2(self._devnull, 1)
        os.dup2(self._devnull, 2)
        return self

    def __exit__(self, *exc):
        os.dup2(self._fd1, 1)
        os.dup2(self._fd2, 2)
        os.close(self._fd1)
        os.close(self._fd2)
        os.close(self._devnull)


def zonotope_contains(gens, e, tol=1e-9):
    """Exact zonotope membership: exists z with G z = e, ||z||_inf <= 1.

    Mirrors Zonotope::contains in tube_mpc_cbf_solver.cpp (a feasibility LP).
    """
    m = gens.shape[1]
    if np.all(np.abs(e) < 1e-12):
        return True
    A = np.vstack([gens, -gens, np.eye(m), -np.eye(m)])
    b = np.concatenate([e, -e, np.ones(m), np.ones(m)])
    b[: 2 * gens.shape[0]] += tol
    res = linprog(np.zeros(m), A_ub=A, b_ub=b, bounds=[(-1.0, 1.0)] * m,
                  method="highs")
    return res.status == 0


class TubeZState:
    """z_previous / v_previous storage (C++ Impl::z_previous, v_guess)."""

    def __init__(self):
        self.z_prev = None
        self.v_prev = None


def pin_stage0(solver, z0):
    """C++ solve() step 5: lbx = ubx = z0 (idxbx covers all four axes, so the
    per-stage bounds pin every component)."""
    solver.set(0, "lbx", z0)
    solver.set(0, "ubx", z0)
    solver.set(0, "x", z0)


def make_tube_solver(tighten_mode="support_function", max_iter=20):
    """Build a tube MPC-CBF solver with the C++ initialize() tightenings.

    Input bounds U (-) K Omega and velocity state bounds X (-) Omega are
    applied exactly as TubeMpcCbfSolver::initialize() does; c_u is the support
    h_{K Omega}(e_i). Position axes get the +-1e9 sentinel.
    """
    ocp = build_tube_ocp(model_name="double_integrator_2d", horizon=N, dt=DT,
                         n_obstacles=N_OBSTACLES, tighten_mode=tighten_mode,
                         use_rti=False, max_sqp_iterations=max_iter)
    c_u = np.abs(K @ omega_gens).sum(axis=1)
    ocp.constraints.lbu = -(1.0 - c_u)
    ocp.constraints.ubu = (1.0 - c_u)
    v_lo = -(2.0 - half[2])
    v_hi = (2.0 - half[2])
    ocp.constraints.idxbx = np.arange(4)
    ocp.constraints.lbx = np.array([-1.0e9, -1.0e9, v_lo, v_lo])
    ocp.constraints.ubx = np.array([1.0e9, 1.0e9, v_hi, v_hi])
    return AcadosOcpSolver(ocp)


def c_tighten(z, half):
    """Barrier tightening margin c_j(z) = |dh/dx| . Omega half-width (support
    function of Omega in the -grad h direction, exact for the 2-norm barrier)."""
    d = 2.0 * (z[:2] - OBST_POS[:2])
    return float(half[0] * abs(d[0]) + half[1] * abs(d[1]))


def c_zero(z, half):
    """Untightened ablation (the nominal controller): no margin."""
    return 0.0


def tube_solve(solver, x0, c_fn, half, gens, zs, warm="shift", check_z0=True):
    """One tube solve mirroring C++ TubeMpcCbfSolver::solve() semantics.

    check_z0=True runs the exact zonotope-membership LP for the z0 policy (the
    faithful mirror of C++ Zonotope::contains). The magnitude sweep and the
    zero-disturbance run pass check_z0=False: for the tube the error never
    leaves Omega so the LP is the identity, and it avoids ~180k LPs.
    """
    with _silence_io():
        have_z = zs.z_prev is not None
        # z0 policy (08_TUBE.md 8.5): continue the previous plan when the error
        # is contained; the true state lives in x0 in z0 + Omega.
        if have_z and (not check_z0 or zonotope_contains(gens, x0 - zs.z_prev[:, 1])):
            z0 = zs.z_prev[:, 1]
        else:
            z0 = x0
        # shifted warm start: z_guess[:,k] = z_prev[:,k+1], tail held
        if have_z:
            z_guess = np.empty((4, N + 1))
            z_guess[:, :N] = zs.z_prev[:, 1:]
            z_guess[:, N] = zs.z_prev[:, N]
        else:
            z_guess = None
        rf._set_reference(solver, GOAL, "fixed_decay")
        pin_stage0(solver, z0)
        if warm == "coast" or z_guess is None:
            rf._coast_warm_start(solver, z0)
        else:
            for k in range(N + 1):
                solver.set(k, "x", z_guess[:, k])
            for k in range(N):
                solver.set(k, "u", zs.v_prev[:, k])
        for k in range(N + 1):
            z_nom = z0 if (k == 0 or z_guess is None) else z_guess[:, k]
            cj = c_fn(z_nom, half)
            p = np.zeros(tube_param_count(N_OBSTACLES))
            p[0:3] = OBST_POS
            p[3:6] = 0.0
            p[6] = R_EFF
            for j in range(1, N_OBSTACLES):
                p[7 * j:7 * j + 3] = 1.0e6
                p[7 * j + 6] = 0.0
            p[7 * N_OBSTACLES] = GAMMA
            p[7 * N_OBSTACLES + 1] = cj
            solver.set(k, "p", p)
        status = solver.solve()
        z_pred = np.column_stack([np.asarray(solver.get(k, "x")) for k in range(N + 1)])
        v_pred = np.column_stack([np.asarray(solver.get(k, "u")) for k in range(N)])
        zs.z_prev = z_pred
        zs.v_prev = v_pred
        v0 = np.array(solver.get(0, "u"))[:2]
        return status, v0, z0, z_pred


def run_closed_loop(solver, c_fn, half, gens, K_mat, n_steps, w_fn,
                    collect_e=False, check_z0=True):
    """Closed-loop rollout: u_applied = clip(v0 + K (x - z0), u_min, u_max)."""
    zs = TubeZState()
    x = np.zeros(4)
    min_h = 1e9
    violations = 0
    cost = 0.0
    e_samples = []
    statuses = {}
    for step in range(n_steps):
        status, v0, z0, z_pred = tube_solve(solver, x, c_fn, half, gens, zs,
                                            check_z0=check_z0)
        statuses[status] = statuses.get(status, 0) + 1
        u_applied = np.clip(v0 + K_mat @ (x - z0), -1.0, 1.0)
        if status not in (0, 2) or not np.all(np.isfinite(u_applied)):
            violations += 1
            break
        w = w_fn(step)
        x_next = advance(x, u_applied, w)
        if collect_e:
            e_samples.append(x_next - z_pred[:, 1])
        h = barrier_h(x)
        if h < 0:
            violations += 1
        min_h = min(min_h, h)
        cost += float(u_applied @ u_applied)
        x = x_next
    return min_h, violations, cost, e_samples, statuses


In [ ]:
# --- Cross-check Omega against compute_offline_sets() ------------------------
# Before plotting anything, re-derive the RPI set from the same LQR gain the
# C++ fixture uses (lqr_Q = [10,10,1,1], lqr_R = [1,1]) and assert agreement to
# 1e-6 (05_CODEGEN.md 5.7). The offline sets are the single source of truth.
K, omega_verts, alpha, s = compute_offline_sets(
    "double_integrator_2d", DT, W_HALF, [10, 10, 1, 1], [1, 1])
half = np.abs(omega_verts).max(axis=0)

# Exact ZOH discretisation of the double integrator (models.double_integrator_2d)
A = np.eye(4)
A[0:2, 2:4] = DT * np.eye(2)
B = np.zeros((4, 2))
B[0:2] = 0.5 * DT * DT * np.eye(2)
B[2:4] = DT * np.eye(2)
rho = max(abs(np.linalg.eigvals(A + B @ K)))

print(f"alpha = {alpha:.16f}, s = {s}")
print(f"Omega hull half-widths = {half}")
print(f"spectral radius(A+BK) = {rho:.16f}")
assert abs(alpha - 0.0028623776978634274) < 1e-6
assert s == 49
assert np.allclose(half, [0.11553617, 0.11553617, 0.16415113, 0.16415113], atol=1e-6)
assert abs(rho - 0.8735317587765347) < 1e-6

# Re-derive the generator matrix directly and assert it matches the offline path.
alpha2, s2, omega_gens = _rakovic_rpi(A + B @ K, W_HALF, 1e-3, 100)
assert alpha2 == alpha and s2 == s
print(f"Omega generator matrix shape = {omega_gens.shape}")
print("cross-check OK")


In [ ]:
# --- 1. The RPI set: 10 000 sampled errors, 100 % containment ---------------
# 50 seeds x 200 steps of random disturbance at the DESIGN w_max = 1.0. The
# closed-loop error e_k = x_{k+1} - z_{k+1|k} must stay in Omega every step;
# if it escapes, the RPI computation is wrong and everything downstream is
# decoration (12_ANALYSIS.md 12.5).
solver = make_tube_solver()  # support_function tightening
min_h_all = 1e9
viol_total = 0
e_all = []
for seed in range(50):
    rngs = np.random.default_rng(RNG_SEED ^ seed)

    def w_rand(step, rng=rngs):
        return W_HALF * rng.uniform(-1, 1, 4)

    mh, v, cost, e, st = run_closed_loop(solver, c_tighten, half, omega_gens, K,
                                         200, w_rand, collect_e=True)
    min_h_all = min(min_h_all, mh)
    viol_total += v
    e_all.extend(e)
    if seed % 10 == 0:
        print(f"seed {seed}: min_h {mh:.6f}, viol {v}, statuses {st}", flush=True)

e_arr = np.array(e_all)
contained_box = np.all(np.abs(e_arr) <= half + 1e-9, axis=1).mean()
lp_ok = sum(zonotope_contains(omega_gens, e) for e in e_arr) / len(e_arr)
print(f"tube random w_max=1: 50x200 -> min h {min_h_all:.6f}, viol {viol_total}, "
      f"box containment {contained_box:.5f}, zonotope containment {lp_ok:.5f} "
      f"({len(e_arr)} samples)")
assert viol_total == 0 and min_h_all >= 0.0
assert contained_box == 1.0 and lp_ok == 1.0


# --- plot: Omega projected onto (px,py) and (vx,vy), errors overlaid ----------
def zonotope_boundary(gens2d, n=256):
    """Exact boundary of a 2-D centred zonotope via its support function
    h_Z(d) = ||G^T d||_1 (closed form -- the same support used for tightening)."""
    th = np.linspace(0.0, 2.0 * np.pi, n, endpoint=False)
    d = np.stack([np.cos(th), np.sin(th)])          # (2, n)
    supp = np.abs(gens2d.T @ d).sum(axis=0)         # (n,)
    return d * supp                                  # (2, n) boundary points

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, idx in zip(axes, [(0, 1), (2, 3)]):
    bd = zonotope_boundary(omega_gens[np.array(idx), :])
    ax.fill(bd[0], bd[1], color="C0", alpha=0.15, label="Omega")
    ax.plot(bd[0], bd[1], color="C0", lw=1.5)
    ax.scatter(e_arr[:, idx[0]], e_arr[:, idx[1]], s=1.0, color="k",
               alpha=0.3, label="closed-loop errors")
    ax.set_aspect("equal")
    ax.set_xlabel(f"e[{idx[0]}]")
    ax.set_ylabel(f"e[{idx[1]}]")
    ax.set_title(f"Omega projection on (e[{idx[0]}], e[{idx[1]}])")
    ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
fig.savefig(FIGDIR / "rpi_set.png", dpi=150)
print(f"wrote {FIGDIR / 'rpi_set.png'}")


# --- bonus: the guarantee in one line -- tube vs untightened under worst-case w
solver2 = make_tube_solver()
zs = TubeZState()
x = np.zeros(4)
min_h = 1e9
viol = 0
for step in range(200):
    status, v0, z0, _ = tube_solve(solver2, x, c_tighten, half, omega_gens, zs)
    u_applied = np.clip(v0 + K @ (x - z0), -1.0, 1.0)
    if status not in (0, 2) or not np.all(np.isfinite(u_applied)):
        viol += 1
        break
    x = advance(x, u_applied, worst_case_w(x))
    min_h = min(min_h, barrier_h(x))
print(f"tube worst-case w: min h {min_h:.6f}, unusable steps {viol}")
assert viol == 0 and min_h >= 0.0

solver_n = make_tube_solver(tighten_mode="none")
zs = TubeZState()
x = np.zeros(4)
min_h = 1e9
viol = 0
for step in range(200):
    status, v0, z0, _ = tube_solve(solver_n, x, c_zero, half, omega_gens, zs)
    u_applied = np.clip(v0 + K @ (x - z0), -1.0, 1.0)
    if status not in (0, 2) or not np.all(np.isfinite(u_applied)):
        viol += 1
        break
    x = advance(x, u_applied, worst_case_w(x))
    min_h = min(min_h, barrier_h(x))
print(f"kNone worst-case w: min h {min_h:.6f}, unusable steps {viol}")
assert min_h < 0.0, "ablation stayed safe under worst-case w"


In [ ]:
# --- 2. Magnitude sweep: nominal vs tube across w_max ------------------------
# 50 seeds x 200 steps per w_max, geometric ladder. The tube must hold
# min_k h >= 0 up to the design w_max = 1.0 AND THEN ALSO FAIL at larger
# magnitudes: a tube that never fails means W was specified larger than the
# disturbance actually applied -- fix the experiment, not the plot.
LADDER = [0.5, 1.0, 1.25, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0]
rows = []
st = make_tube_solver()                            # support_function + c_tighten
sn = make_tube_solver(tighten_mode="none")         # untightened ablation + c_zero
for wmax in LADDER:
    v_t = v_n = 0
    mh_t = 1e9
    mh_n = 1e9
    cost_t = cost_n = 0.0
    for seed in range(50):
        rngs = np.random.default_rng(RNG_SEED ^ seed)

        def w_rand(step, rng=rngs):
            return wmax * W_HALF * rng.uniform(-1, 1, 4)

        mt, vt, ct, _, _ = run_closed_loop(st, c_tighten, half, omega_gens, K,
                                           200, w_rand, check_z0=False)
        mn, vn, cn, _, _ = run_closed_loop(sn, c_zero, half, omega_gens, K,
                                           200, w_rand, check_z0=False)
        v_t += vt
        v_n += vn
        mh_t = min(mh_t, mt)
        mh_n = min(mh_n, mn)
        cost_t += ct
        cost_n += cn
    rows.append((wmax, mh_t, v_t, cost_t, mh_n, v_n, cost_n))
    print(f"w_max={wmax:4.2f}: tube min_h {mh_t:+.5f} viol {v_t}/50 | "
          f"nominal min_h {mh_n:+.5f} viol {v_n}/50", flush=True)

with open(HERE / "results_disturbance.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["w_max", "tube_min_h", "tube_violations", "tube_cost",
                "nominal_min_h", "nominal_violations", "nominal_cost"])
    w.writerows(rows)
print(f"wrote {HERE / 'results_disturbance.csv'}")

# asserts on the expected SHAPE (exact values are the prints above)
arr = np.array(rows)
wmaxs = arr[:, 0]
tube_h = arr[:, 1]
tube_v = arr[:, 2]
nom_h = arr[:, 4]
nom_v = arr[:, 5]
design = {w: (th, nh, tv, nv) for w, th, tv, nh, nv in
          zip(wmaxs, tube_h, tube_v, nom_h, nom_v)}
# design w_max = 1.0: tube holds, nominal violates
assert design[1.0][0] >= 0.0 and design[1.0][2] == 0
assert design[1.0][1] < 0.0 and design[1.0][3] > 0
# tube holds through 4.0, then also fails at 6.0 and 8.0
assert design[4.0][0] >= 0.0 and design[4.0][2] == 0
assert design[6.0][0] < 0.0 and design[6.0][2] > 0
assert design[8.0][0] < 0.0 and design[8.0][2] > 0
# nominal violates at every magnitude
assert np.all(nom_h < 0.0) and np.all(nom_v > 0)


# --- plot: min_h and violation rate vs w_max (log-x) -------------------------
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].axhline(0.0, color="k", lw=0.8, ls="--")
axes[0].plot(wmaxs, tube_h, "o-", label="tube (support_function)")
axes[0].plot(wmaxs, nom_h, "s-", label="nominal (untightened)")
axes[0].set_xscale("log")
axes[0].set_xlabel("w_max (disturbance magnitude)")
axes[0].set_ylabel("min_k h (over 50 seeds)")
axes[0].legend()
axes[1].plot(wmaxs, tube_v / 50, "o-", label="tube")
axes[1].plot(wmaxs, nom_v / 50, "s-", label="nominal")
axes[1].set_xscale("log")
axes[1].set_xlabel("w_max (disturbance magnitude)")
axes[1].set_ylabel("violation rate")
axes[1].legend()
fig.tight_layout()
fig.savefig(FIGDIR / "robustness_sweep.png", dpi=150)
print(f"wrote {FIGDIR / 'robustness_sweep.png'}")


In [ ]:
# --- 3. Conservatism cost at zero disturbance --------------------------------
# The tube's guarantee is set-containment e_k in Omega, not trajectory
# optimality. At zero disturbance it still solves a tightened problem, which
# costs path length and time-to-goal versus the nominal controller. This is
# the honest counterweight to the safety claim, reported in the README.
def run_w0(solver, c_fn, n_steps=400):
    zs = TubeZState()
    x = np.zeros(4)
    path = 0.0
    traj = [x.copy()]
    steps_to_goal = None
    min_h = 1e9
    for step in range(n_steps):
        status, v0, z0, z_pred = tube_solve(solver, x, c_fn, half, omega_gens, zs,
                                            check_z0=False)
        u_applied = np.clip(v0 + K @ (x - z0), -1.0, 1.0)
        if status not in (0, 2) or not np.all(np.isfinite(u_applied)):
            break
        x_next = advance(x, u_applied, np.zeros(4))
        path += float(np.linalg.norm(x_next[:2] - x[:2]))
        h = barrier_h(x)
        min_h = min(min_h, h)
        x = x_next
        traj.append(x.copy())
        if steps_to_goal is None and np.linalg.norm(x[:2] - GOAL[:2]) < 0.05:
            steps_to_goal = step + 1
    return path, steps_to_goal, np.array(traj), min_h

pt, stg_t, traj_t, mh_t = run_w0(st, c_tighten)
pn, stg_n, traj_n, mh_n = run_w0(sn, c_zero)
overhead = (pt / pn - 1.0) * 100.0
print(f"zero-disturbance: tube path {pt:.4f} steps_to_goal {stg_t} "
      f"min_h {mh_t:+.5f} | nominal path {pn:.4f} steps_to_goal {stg_n} "
      f"min_h {mh_n:+.5f}")
print(f"path overhead {overhead:.2f}%")

# both controllers reach the goal; the tube pays a path/time premium
assert stg_t is not None and stg_n is not None
assert stg_t == 48 and stg_n == 43
assert abs(pt - 2.0501) < 0.01 and abs(pn - 1.8261) < 0.01
assert abs(overhead - 12.27) < 0.1
assert mh_t >= 0.0 and mh_n >= 0.0


## What the numbers say — and what they don't

Three measurements, three honest conclusions:

1. **The RPI certificate holds.** Every one of the 10 000 sampled closed-loop
   errors lies inside `Omega` (asserted 100 % containment, both the axis hull
   `|e| <= half` and the exact zonotope LP). The tube's guarantee —
   `e_k in Omega` for all `k` — is a *set-containment* certificate, not a
   trajectory promise. It says nothing about nominal safety by itself; that is
   measured next.

2. **The tube holds — and then also fails.** Up to the design `w_max = 1.0` the
   tube keeps `min_k h >= 0` with zero violations while the untightened
   (nominal) controller violates at every magnitude. Pushed to `w_max = 4.0`
   the tube still holds (`min h ≈ +0.021`), but at `6.0` it fails
   (`min h ≈ -0.071`, 7/50 seeds) and at `8.0` it fails harder
   (`min h ≈ -0.120`, 39/50). This is the *required* shape: a tube that never
   fails would mean `W` was specified larger than the disturbance actually
   applied — the experiment, not the plot, would need fixing.

3. **Safety costs ~12 % of the path.** At zero disturbance the tube reaches the
   goal in 48 steps (path 2.05) versus the nominal controller's 43 steps
   (path 1.83): a ~12.3 % path overhead and five extra steps, because the tube
   solves a tightened problem even when no disturbance is present. That number
   is the honest counterweight to the safety claim, and it is the one quoted in
   the README.

**On the cost weights.** The C++ test fixture (`test_tube_mpc_robustness.cpp`)
sets `Qf = Q = [10,10,1,1]` for its safety tests, but the shipped configuration
(`mpc_cbf_params.yaml`) uses `Qf = 10 Q = [100,100,10,10]`. This notebook uses
the shipped weights, because item 3 needs a well-defined time-to-goal: with
`Qf = Q` the tube idles at `|x| ≈ 0.145` and never reaches the goal. The safety
and containment numbers (items 1–2) are insensitive to this choice.
